In [ ]:
from itables import init_notebook_mode, show
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import importlib
import aacbr 

init_notebook_mode(all_interactive=True)

In [ ]:
def reload_imports():
    importlib.reload(aacbr)

## Data Set

In [ ]:
# from ucimlrepo import fetch_ucirepo 
  
# # fetch dataset 
# connectionist_bench_sonar_mines_vs_rocks = fetch_ucirepo(id=151) 
  
# # data (as pandas dataframes) 
# X = connectionist_bench_sonar_mines_vs_rocks.data.features 
# y = connectionist_bench_sonar_mines_vs_rocks.data.targets 

data = pd.read_csv('data/connectionist-bench-sonar-mines-vs-rocks/sonar.all-data')

data = data.values

X = np.array(data[:, :-1], dtype=np.float32)
y = np.array(data[:, -1])

show(X)
print(np.unique(y))



In [ ]:
encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)


In [ ]:
print(encoder.classes_)
print(y)
print(type(y))

## Train Model

### Split into Training and Test Sets

In [ ]:
SEED = 42

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=SEED)
print(f"Test Size:  {len(X_test)}")
print(f"Train Size:  {len(X_train)}")
print(f"Validation Size:  {len(X_val)}")



In [ ]:
print(X_train)

### Build AF

In [ ]:
# TODO: Consider more sophisticated orders

# Compare against the average for each column
means = X_train.mean(axis=0)
std = X_train.std(axis=0)

STD_PARAM = 2

def binarise_by_normal(case):
    return np.where(np.abs(case - means) <= STD_PARAM*std, 0, 1)


def strictsuperset(a, b):

    if b.ndim == 1:
        b = np.expand_dims(b, axis = 0)

    anb = a & b
    return np.logical_and(np.all(anb == b, axis = -1), np.logical_not(np.all(anb == a, axis = -1)))


In [ ]:
def to_stds(case):

    case = case - means

    conditions = [case >=  3 * std, case >=  2 * std, case >=  1.5 * std, 
                  case > -1.5 * std,
                  case <= -3 * std, case <= -2 * std, case <= -1.5 * std ]

    # choices = [3, 2, 1, 0, 3, 2, 1]
    # choices = [3, 2, 0, 0, 3, 2, 0]

    choices = [3, 2, 1, 0, -3, -2, -1]
    # choices = [3, 2, 0, 0, -3, -2, 0]
    return np.select(conditions, choices)
    


def strictstdcomp(a, b):
    if b.ndim == 1:
        b = np.expand_dims(b, axis = 0)
    same_sign_or_zero = np.all(np.logical_or(np.sign(a) == np.sign(b), np.logical_or(a == 0, b == 0)), axis = -1)
    further_from_zero = np.all(np.abs(a) >= np.abs(b), axis = -1)
    noteq = np.any(a != b, axis = -1)

    return np.logical_and(same_sign_or_zero, np.logical_and(further_from_zero, noteq))


In [ ]:
COMPARISON_FUNC = strictsuperset
PREPROCESS_FUNC = binarise_by_normal 
# COMPARISON_FUNC = strictstdcomp
# PREPROCESS_FUNC = to_stds 

In [ ]:
DEFAULT_OUTCOME = 1
DEFAULT_CASE = means.copy()
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

In [ ]:
reload_imports()
def run_model(X_train, y_train, X_test, y_test, print_graph=False, print_matrix=False):
    X_train = PREPROCESS_FUNC(X_train)
    X_test = PREPROCESS_FUNC(X_test)
    default_case = PREPROCESS_FUNC(DEFAULT_CASE)
    
    model = aacbr.AACBR(X_train, y_train, COMPARISON_FUNC, default_case, [DEFAULT_OUTCOME])
    predicted = model(X_test)

    if print_graph:
        model.show_graph_with_labels()

    if print_matrix:
        model.show_matrix()
    
    return([
        accuracy_score(y_test, predicted),
        precision_score(y_test, predicted),
        recall_score(y_test, predicted),
        f1_score(y_test, predicted)
    ])

### Cross-Validation

In [ ]:
reload_imports()
metrics = []
for fold, (train_index,  val_index) in enumerate(kf.split(X_train)):
    training_instances = X_train[train_index]
    training_labels = y_train[train_index]
    validation_instances = X_train[val_index]
    validation_labels = y_train[val_index]


    metrics.append(
        run_model(training_instances, training_labels, validation_instances, validation_labels)
    )

print("Accuracy, Precision, Recall, F1")
print(np.mean(metrics, axis=0))
# for metric in metrics:
#     print(metric)


In [ ]:
reload_imports()
results = run_model(X_train, y_train, X_val, y_val, print_matrix=True)
print("Accuracy, Precision, Recall, F1")
print(np.squeeze(results))

### Test Set

In [ ]:
# reload_imports()
# print("Accuracy, Precision, Recall, F1")
# run_model(X_train_full, y_train_full, X_test, y_test, show_graph=False)